# Docker Desktop

In this notebook, we:

- Ensure Docker Desktop's Kubernetes cluster is working.
- Ensure the `kubectl` tool is installed and working.
- Install `Helm` on our local computer.
- Install the `metrics server` in the Kubernetes cluster.
- Install the `dashboard` in the Kubernetes cluster.

# Install and Configure Docker Desktop (if not already done)

### Docker Desktop has a Kubernetes distribution which can be used for local development purposes.

- Documentation: https://docs.docker.com/desktop

- Install:

  - **Windows**: https://docs.docker.com/desktop/setup/install/windows-install
  - **MacOS** https://docs.docker.com/desktop/setup/install/mac-install
  - **Ubuntu/WSL2**: https://docs.docker.com/desktop/setup/install/linux

- Run:

  - When you have installed Docker Desktop, make sure you start the application.
  - Ensure you see the text `Engine running` in Docker Desktop's task bar (bottom left corner).  

  <img src="notebook_images/docker_desktop.png" width="800" style="padding-bottom: 1em;" />

- Configure:

  - You can configure Docker Desktop to automatically start when you boot your computer.
  - Click the "cogwheel" icon in the top right.
  - Select the `General` tab.
  - Check the checkbox `Start Docker Desktop when you sign in to your computer`.
  - Click the `Apply & restart` button.

  <img src="notebook_images/docker_desktop_settings_general.png" width="800" style="padding-bottom: 1em;" />
  
- Enable Kubernetes:

  - Enable Docker Desktop's built-in Kubernetes cluster for local development on your computer.
  - Click the "cogwheel" icon in the top right.
  - Select the `Kubernetes` tab.
  - Make sure the toggle `Enable kubernetes` is on.
  - Click the `Apply & restart` button.
  - When Docker Desktop restarts, esure you see the text `Kubernetes running` in Docker Desktop's task bar (bottom center).
  
  <img src="notebook_images/docker_desktop_settings_kubernetes.png" width="800" style="padding-bottom: 1em;" />

# Install `kubectl` (if not already done)

### Kubectl is the Kubernetes CLI tool used to communicate with a Kubernetes cluster.

- Install: https://kubernetes.io/docs/tasks/tools
- Verify: Run the cell below which should print-out a `kubectl` version.

In [4]:
!kubectl version

Client Version: v1.30.5
Kustomize Version: v5.0.4-0.20230601165947-6ce0bf390ce3
Server Version: v1.30.5


## Show `kubectl` help

- `kubectl --help` shows what commands kubectl can run.
- `kubectl <command> --help` shows additional help for the command `<command>`.
  - E.g. `kubectl apply --help` shows additional help for the command `apply`.

In [8]:
#!kubectl apply --help
!kubectl --help

kubectl controls the Kubernetes cluster manager.

 Find more information at: https://kubernetes.io/docs/reference/kubectl/

Basic Commands (Beginner):
  create          Create a resource from a file or from stdin
  expose          Take a replication controller, service, deployment or pod and expose it as a new Kubernetes service
  run             Run a particular image on the cluster
  set             Set specific features on objects

Basic Commands (Intermediate):
  explain         Get documentation for a resource
  get             Display one or many resources
  edit            Edit a resource on the server
  delete          Delete resources by file names, stdin, resources and names, or by resources and label selector

Deploy Commands:
  rollout         Manage the rollout of a resource
  scale           Set a new size for a deployment, replica set, or replication controller
  autoscale       Auto-scale a deployment, replica set, stateful set, or replication controller

Cluster Manage

## Set `kubectl`'s context to Docker Desktop's Kubernetes cluster

In [6]:
!kubectl config use-context docker-desktop
!kubectl config current-context

Switched to context "docker-desktop".
docker-desktop


## List Docker Desktop Kubernetes nodes

- Docker Desktop's Kubernetes cluster runs in one Virtual Machine, which is the only node available.
- The node is a `control plane` node (called `docker-desktop` below), but can run workloads, just like a `worker` node.

In [20]:
!kubectl get nodes

NAME             STATUS   ROLES           AGE   VERSION
docker-desktop   Ready    control-plane   24d   v1.30.5


## List the Kubernetes Processes Running in Docker Desktop Kubernetes VM

**Note! The cell below will only work on Windows and Linux (not Mac)**

Let's print out the Kubernetes-specific processes running in Docker Desktop's Kubernetes VM.

```bash
# containerd      : this is the container runtime engine that manages containers on a node.
# etcd            : this is the etcd key-value database that stores a Kubernetes cluster's state ("single source of truth").
# kube-apiserver  : this is the kube-apiserver that communicates with the etcd database and exposes a REST API.
# kube-controller : this is the kube-controller-manager that controls various resources' controller managers.
# kube-proxy      : this is the kube-proxy that maintains a network routing table and maps Service IPs Pod IPs.
# kube-scheduler  : this is kube-scheduler that schedules new resources on specific nodes.
# kubelet         : this is the kubelet that communicates with the kube-apiserver and manages pods on a node.
```

In [28]:
#screen ~/Library/Containers/com.docker.docker/Data/vms/0/tty
!wsl -d docker-desktop -- sh -c "ps -e -o comm | grep -i -E 'etcd|kube|containerd$' | grep -v 'kube-vpnkit-for' | sort"

containerd
etcd
kube-apiserver
kube-controller
kube-proxy
kube-scheduler
kubelet


## Install `Helm`

[Helm](https://helm.sh/docs) is a package manager for Kubernetes.

- We will use Helm to install the `metrics server` and the `dashboard` into our Kubernetes cluster.

Install [Helm](https://helm.sh/docs/intro/install):

- **Windows**: `winget install Helm.Helm`
- **MacOS**: `brew install helm`
- **Linux**: see https://helm.sh/docs/intro/install

Verify: Run the cell below to verify Helm is installed (it should print out the Helm version).

In [2]:
!helm version

version.BuildInfo{Version:"v3.17.0", GitCommit:"301108edc7ac2a8ba79e4ebf5701b0b6ce6a31e4", GitTreeState:"clean", GoVersion:"go1.23.4"}


## Install the `metrics server` and `dashboard`

The [Metrics Server](https://github.com/kubernetes-sigs/metrics-server) is a Kubernetes application used to scrape metrics from a Kubernetes cluster.

- We can use it to monitor server (node) and pod resource utilization (CPU, RAM, Storage, etc.).
- It's also a prerequisite for horizontal autoscaling which we will also cover in this workshop.

The Kubernetes [Dashboard](https://github.com/kubernetes/dashboard) is a Kubernetes application with a Web UI for managing Kubernetes clusters.

- It's a great tool that complements the `kubectl` CLI with a web-based graphical user interface.

In the cell below, we use `Helm` (the Kubernetes package manager) to install the `metrics server` and the `dashboard`.

In [3]:
# Add the metrics-server and the kubernetes-dashboard to the local Helm repository
!helm repo rm kubernetes-dashboard
!helm repo rm metrics-server
!helm repo add metrics-server https://kubernetes-sigs.github.io/metrics-server/
!helm repo add kubernetes-dashboard https://kubernetes.github.io/dashboard/
!helm repo update
!helm repo ls

# Deploy the Helm Chart for the metrics-server and the kubernetes-dashboard to Docker Desktop's Kubernetes cluster
!helm uninstall kubernetes-dashboard --namespace kubernetes-dashboard
!helm uninstall metrics-server --namespace kube-system
!helm upgrade --install metrics-server metrics-server/metrics-server --set args="{--kubelet-insecure-tls}" --create-namespace --namespace kube-system
!helm upgrade --install kubernetes-dashboard kubernetes-dashboard/kubernetes-dashboard --create-namespace --namespace kubernetes-dashboard
!helm list -A

"kubernetes-dashboard" has been removed from your repositories
"metrics-server" has been removed from your repositories
"metrics-server" has been added to your repositories
"kubernetes-dashboard" has been added to your repositories
Hang tight while we grab the latest from your chart repositories...
...Successfully got an update from the "metrics-server" chart repository
...Successfully got an update from the "kubernetes-dashboard" chart repository
Update Complete. ⎈Happy Helming!⎈
NAME                	URL                                              
metrics-server      	https://kubernetes-sigs.github.io/metrics-server/
kubernetes-dashboard	https://kubernetes.github.io/dashboard/          
release "kubernetes-dashboard" uninstalled
release "metrics-server" uninstalled
Release "metrics-server" does not exist. Installing it now.
NAME: metrics-server
LAST DEPLOYED: Sun Jan 26 08:21:04 2025
NAMESPACE: kube-system
STATUS: deployed
REVISION: 1
TEST SUITE: None
NOTES:
************************

### Verify the `metrics-server` was installed successfully.

- The `kubectl top nodes` command returns metrics (cpu and memory utilization) for the Kubernetes nodes.
- The `kubectl top pods -A` command returns metrics for the Pods in all namespaces (`-A`).

In [4]:
!kubectl top nodes
!kubectl top pods -A

NAME             CPU(cores)   CPU%   MEMORY(bytes)   MEMORY%   
docker-desktop   256m         3%     2904Mi          37%       
NAMESPACE              NAME                                                    CPU(cores)   MEMORY(bytes)   
kube-system            coredns-55cb58b774-67kt4                                3m           43Mi            
kube-system            coredns-55cb58b774-lqndz                                3m           39Mi            
kube-system            etcd-docker-desktop                                     31m          371Mi           
kube-system            kube-apiserver-docker-desktop                           41m          329Mi           
kube-system            kube-controller-manager-docker-desktop                  34m          142Mi           
kube-system            kube-proxy-9d8nz                                        1m           69Mi            
kube-system            kube-scheduler-docker-desktop                           5m           72Mi            


### Create a Service Account, Cluster Role Binding, and Token for the `dashboard`

- In order to access the `dashboard` we need to use Kubernetes' Role Based Access Control (RBAC).
  - Kubernetes RBAC is a topic which is out-of-scope for this course.
- We also need to generate a Bearer Token that we can use to login to the `dashboard`.

Execute the cell below, and copy the token that is returned on the last line in the output.

- We will need this token to login to the `dashboard`.

In [5]:
!kubectl create serviceaccount kubeadmin -n kubernetes-dashboard
!kubectl create clusterrolebinding kubeadmin-binding --clusterrole=cluster-admin --serviceaccount=kubernetes-dashboard:kubeadmin
!kubectl -n kubernetes-dashboard create token kubeadmin

error: failed to create serviceaccount: serviceaccounts "kubeadmin" already exists
error: failed to create clusterrolebinding: clusterrolebindings.rbac.authorization.k8s.io "kubeadmin-binding" already exists


eyJhbGciOiJSUzI1NiIsImtpZCI6IkxQdURCQVhyZEhJQ3F5ZE96RmtuRzd6WVlHQXBsOVFxdUhTU3JFQVVLUjgifQ.eyJhdWQiOlsiaHR0cHM6Ly9rdWJlcm5ldGVzLmRlZmF1bHQuc3ZjLmNsdXN0ZXIubG9jYWwiXSwiZXhwIjoxNzM3ODgwMTU4LCJpYXQiOjE3Mzc4NzY1NTgsImlzcyI6Imh0dHBzOi8va3ViZXJuZXRlcy5kZWZhdWx0LnN2Yy5jbHVzdGVyLmxvY2FsIiwianRpIjoiOTczMTczZTItZjdjZC00OTk4LWJkYmMtZjVjZGZiNDA2YjU2Iiwia3ViZXJuZXRlcy5pbyI6eyJuYW1lc3BhY2UiOiJrdWJlcm5ldGVzLWRhc2hib2FyZCIsInNlcnZpY2VhY2NvdW50Ijp7Im5hbWUiOiJrdWJlYWRtaW4iLCJ1aWQiOiIwMGY5YzBkNS01MDRmLTQxYjAtOTNmYS1mZGQzNzEzMGMwNzUifX0sIm5iZiI6MTczNzg3NjU1OCwic3ViIjoic3lzdGVtOnNlcnZpY2VhY2NvdW50Omt1YmVybmV0ZXMtZGFzaGJvYXJkOmt1YmVhZG1pbiJ9.kSor-5TnJLgy3HE3lSPbSl4nc_pRkVmInnIoWBfCOUNRPLDIehnrNoOudEWuotITtepSPi63wj0345KWSV610mzO-npq516QnCm1YxBLr0QJFTPpo4dW5Ao5bmHMPrH75BFOVpwyMnQZiOT0pLKJkClm71rFXWUMT6FgNC3OWdSHa71fzBPQjl1VvjzycjMZGlxxdL8lmD7iSZ5mgUW7Ub2ld7fHSJD_FHSmUDUaTA4m4OlJacwQPo5uvyUyq1Rxp3sW5y6hcfELb17_DogcA8V9iNA0pJ0l0JV1LtTsMzL7u3MdSeIh5qRtgiM2Z8OPFAfiwz00lTOHGgcJOJ--ZA


## Open the Dashboard

- Run the following command in a separate terminal (and keep it open) to open a **proxy to the dashboard**:
  - `kubectl -n kubernetes-dashboard port-forward svc/kubernetes-dashboard-kong-proxy 8443:443`
- Visit: https://localhost:8443
- Enter the `Bearer Token` from above and click `Sign In` to login to the `dashboard`.
- You should now see the `dashboard` below.

<img src="notebook_images/dashboard.png" alt="dashboard" width="1000" height="500"/>

Under **Workloads** in the left margin
- Click on **Deployments** to see the cluster's Deployments (you won't see any at the moment).
- Click on **Pods** to see the cluster's Pods (you won't see any at the moment).
- Click on **Replica Sets** to see the cluster's ReplicaSets (you won't see any at the moment).

Under **Service** in the left margin
- Click on **Ingresses** to see the cluster's Ingresses (you won't see any at the moment).
- Click on **Services** to see the cluster's Services (you will see one service here at the moment).
  - The `kubernetes` service is used within the cluster to access the kube-apiserver, so **never delete it**.

Under **Cluster** in the left margin
- Click on **Namespaces** to see the cluster's Namespaces (you will see five namespaces here at the moment).
  - The namespaces `kube-node-lease`, `kube-public` and `kube-system` contain kubernetes system resources, so **never delete them**.
  - The namespace `kubernetes-dashboard` contains resources for the Dashboard.
  - The namesapce `default` is where new resources will be added by default.
- Click on **Nodes** to see the cluster's Nodes (you will see one node here).
  - The `docker-desktop` node is the control plane node (and also acts as a worker node).

Notice the three vertical dots `:` in the far right of every listed resource.
- You can click this and choose
  - `Edit` to edit the YAML for this resouce (when saved, it will update the resource in the cluster).
  - `Delete` to delete the resouce from the cluster.

Notice the `+` icon in the top right of the Web UI.
- You can click this to add a resource to the cluster, e.g. via YAML (when saved, it till add the resouce to the cluster).

Notice the combo box in the top left of the Web UI
- You can use this to choose the namespace that is used to display the various resources above.
  - Choose a specific namespace (e.g. `default`) to only see resources in that namespace.
  - Choose `All namespaces` to see resources in all namesapces.

**Press `Ctrl + C` (`Cmd + C` on Mac) in the terminal when you are done using the `dashboard` to stop and remove the proxy to it.**

Now we are ready to use Docker Desktop's Kubernetes cluster.